# Initial Steps: load packages, files and functions

In [1]:
import urllib
import time
import os
import math
import json
import re
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

In [2]:
cwd=os.getcwd()
cwd_Raw_Data_outputs=os.path.join(cwd,'RawData')#heres where we store freezes of the raw data
# cwd_Figures=os.path.join(cwd,'Figures')#figures and code for generating them can go here
cwd_Output=os.path.join(cwd,'Output Dataframes')

In [3]:
dfSynonym=pd.read_excel(os.path.join(cwd,'Synonyms_filtered_v3.xlsx'),engine="openpyxl")
dfSynonym=dfSynonym.sort_values(by="Symbol")
dfSynonym=dfSynonym.reset_index()
dfSynonym=dfSynonym.drop(columns="index")
dfSynonym.index=dfSynonym["Symbol"]

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\david\\OneDrive\\Desktop\\Forbeck_Reporter_Downloader_and_Raw_Data\\Synonyms_filtered_v3.xlsx'

In [4]:
gene_terms=str("(\"+_gene_+\"[ti])")
CT1_queryPM=str("")
CT1_queryNIH=str("")

In [5]:
def GetAwardAmount(input_String, Lists):
    #function takes three arguments; the Reporter output, the destination where we store results, and an additional list for storing a freeze of the data
    GetResults = input_String.find("\"results\"") #find the part of the output detailing grant award amount, found after the "results" block of the ouput
    ResultsList = input_String[GetResults:].replace("},", "")# each grant's information is separated by curly brackets; splitting along curly brackets divides info from each grant
    ResultsList = (ResultsList.split("{\""))[1:]    #saving the individual grant amount as a an element in a list of grants
    for iGrant in ResultsList:# for each grant returned by the query
        Award_Start = iGrant.find("\"award_amount\":")#find the part detailing award amount
        Award_End = iGrant.find("\"project_start_date\":")#find the part that comes after the award amount
        DirectCost = iGrant.find("\"direct_cost_amt\":")
        direct_End = iGrant.find("\"indirect_cost_amt\":")
        Award_string = iGrant[Award_Start:Award_End].replace(",", "").split(":")[1]# the amount of money for grant will be between the part addressed as award amount and the direct cost amount
        directCost = iGrant[DirectCost:direct_End].replace(",", "").split(':', 1)[1]
        indirectCost = iGrant[direct_End:].replace(",", "").split(':', 1)[1]
        if not Award_string == "null": # for some reason, some grants do not have an award amount stored in NIH Reporter
            Lists[0] = Lists[0] + int(Award_string)
            if not directCost == "null":
                Lists[1]=Lists[1]+int(directCost)
            if not "null" in indirectCost:
                indirectCost=indirectCost.replace("}]}","")
                Lists[2]=Lists[2]+int(indirectCost)
    return Lists

# Search 1: Standard Search with cancer and genes

In [ ]:
url_Pubmed_S1='https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=pubmed&term=(cancer[ti]+AND+(\"+_gene_+\"[ti]))&retmax=20'
Out_DF_S1=pd.DataFrame(columns=["Gene name","Pubs[title]","Pubs[title/abstract]", "Number of Grants[title/abstract]", "Award Amount[title/abstract]", "Number of Grants[title]", "Award Amount[title]", "Synonyms"])
PM_Tiab_raw='Default Search Parameter Raw Data\\Standard Search with cancer and genes\\NCBI PubMed\\Title and abstract data\\'
PM_Ti_raw='Default Search Parameter Raw Data\\Standard Search with cancer and genes\\NCBI PubMed\\Title only data\\'
NIH_Ti_raw='Default Search Parameter Raw Data\\Standard Search with cancer and genes\\NIH RePORTER\\Title only data\\'
NIH_Tiab_raw='Default Search Parameter Raw Data\\Standard Search with cancer and genes\\NIH RePORTER\\Title and abstract data\\'
paramsDefault = {
        "criteria": {
    "advanced_text_search": {"operator": "advanced", "search_field": "projecttitle,abstracttext","search_text": str("cancer AND ")}},
        "include_fields": ["ApplId", "ProjectTitle", "AwardAmount", "DirectCostAmt", "IndirectCostAmt","ProjectStartDate", "ProjectEndDate"],"offset": 0, "limit": 500, }
Out_DF_S1CD=AccessPubMed_and_Reporter_Master_function(Out_DF_S1,url_Pubmed_S1,paramsDefault,PM_Ti_raw,PM_Tiab_raw,NIH_Ti_raw,NIH_Tiab_raw)
print(Out_DF_S1CD)
df=pd.read_csv(os.path.join(cwd_Raw_Data_outputs,"NIH+PM_TP53.csv"), sep=",", header=0)#path for manually downloaded TP53 title/abstract award amount
df=df.iloc[:, :-1]
df["Total"]=df["Total Cost"]+df["Total Cost (Sub Projects)"]
Total_TP53=0
for i in df["Total"]:
    if (i!="" and i!="  "):#NIH stores empty values as spaces for some reason 
        Total_TP53=int(i)+Total_TP53
Out_DF_S1CD.index=Out_DF_S1CD["Gene name"]
Out_DF_S1CD=Out_DF_S1CD.drop(columns="Gene name")
Out_DF_S1CD.loc["TP53"]["Award Amount[title/abstract]"]=int(Total_TP53)
Out_DF_S1CD.to_excel(os.path.join(cwd_Output,"NIH+PM_Data.xlsx"), engine="openpyxl")

ABI1
title block
{'Gene name': 'ABI1', 'Pubs[title]': 6, 'Pubs[title/abstract]': 35, 'Number of Grants[title/abstract]': 15, 'Award Amount[title/abstract]': 3781139, 'Number of Grants[title]': 0, 'Award Amount[title]': 0, 'Synonyms': 'ABI-1|ABLBP4|E3B1|NAP1BP|SSH3BP|SSH3BP1|'}
ABL1
title block
{'Gene name': 'ABL1', 'Pubs[title]': 114, 'Pubs[title/abstract]': 2258, 'Number of Grants[title/abstract]': 1060, 'Award Amount[title/abstract]': 383784914, 'Number of Grants[title]': 17, 'Award Amount[title]': 2089563, 'Synonyms': 'ABL|BCR-ABL|CHDSKM|JTK7|bcr/abl|c-ABL|c-ABL1|p150|v-abl|'}
ABL2
excluded:  ARG
title block
{'Gene name': 'ABL2', 'Pubs[title]': 6, 'Pubs[title/abstract]': 61, 'Number of Grants[title/abstract]': 22, 'Award Amount[title/abstract]': 2335940, 'Number of Grants[title]': 2, 'Award Amount[title]': 76296, 'Synonyms': 'ABLL|'}
ACKR3
title block
{'Gene name': 'ACKR3', 'Pubs[title]': 6, 'Pubs[title/abstract]': 47, 'Number of Grants[title/abstract]': 16, 'Award Amount[title/abst

In [4]:

xlsx_path = Path("Altered Meeting history.xlsx")  # <-- change if needed

# Read sheets (your first sheet name is a bit odd, so grab by index)
xls = pd.ExcelFile(xlsx_path)
participants_df = pd.read_excel(xlsx_path, sheet_name=xls.sheet_names[0])
meetings_df      = pd.read_excel(xlsx_path, sheet_name="Meetings")

participants_df.iloc[ 1:2] = "Targeting Lipid Biology in Cancer"
participants_df.head(5)

,Participant,Meeting,Type,First Name,Last Name,Suffix,Institution,Title
0,"Alison Ringel, PhD",Targeting Lipid Biology in Cancer,Scholar,Alison,Ringel,PhD,NaN,NaN
1,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer
2,"Bart Vanhaesebroeck, PhD",Targeting Lipid Biology in Cancer,Participant,Bart,Vanhaesebroeck,PhD,University College London,Professor of Cell Signaling
3,"Christina Mitchell, MB BS, PhD",Targeting Lipid Biology in Cancer,Participant,Christina,Mitchell,"MB BS, PhD",Monash Institute of Pharmaceutical Sciences,NaN
4,"Neil Vasan, MD, PhD",Targeting Lipid Biology in Cancer,Participant,Neil,Vasan,"MD, PhD",Columbia University,Assistant Professor of Medicine


In [5]:
# Cell 2 — helpers (normalize meeting titles + parse chair names)

def normalize_meeting_title(x: str) -> str:
    """
    Make meeting titles comparable across sheets:
    - cast to str
    - strip leading/trailing whitespace
    - remove surrounding quotes
    - collapse internal whitespace (including newlines)
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    # remove one pair of surrounding quotes if present
    if (len(s) >= 2) and ((s[0] == s[-1]) and s[0] in {"'", '"'}):
        s = s[1:-1].strip()
    s = re.sub(r"\s+", " ", s)  # collapse newlines/tabs/multiple spaces
    return s

def split_chair_names(chairs_cell) -> list[str]:
    """
    Turn the 'Meeting Chairs' cell into a list of chair name strings.
    Handles separators like ';', ',', ' and ', '&', and common ' of ' patterns.
    Keeps credentials as part of the name string (e.g., 'MD, PhD').
    """
    if pd.isna(chairs_cell):
        return []
    s = str(chairs_cell).strip()
    s = re.sub(r"\s+", " ", s)

    # Many entries look like "Name of Institution; Name of Institution"
    # Split primarily on ';' first.
    parts = [p.strip() for p in s.split(";") if p.strip()]

    # Further split each part on " and " / " & " if it contains multiple chairs.
    chairs = []
    for p in parts:
        sub = re.split(r"\s+(?:and|&)\s+", p)
        for item in sub:
            item = item.strip()
            if not item:
                continue
            # Remove trailing institution phrase like " of XYZ" (optional, but helps matching)
            item = re.sub(r"\s+of\s+.+$", "", item).strip()
            chairs.append(item)

    # de-dup while preserving order
    seen = set()
    out = []
    for c in chairs:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

In [6]:
meetings_df = meetings_df.copy()
participants_df = participants_df.copy()

meetings_df["MeetingTopic_norm"] = meetings_df["Meeting Topic"].map(normalize_meeting_title)
participants_df["Meeting_norm"]  = participants_df["Meeting"].map(normalize_meeting_title)

# Map normalized meeting topic -> year (if duplicates exist, keep the first non-null year)
meeting_to_year = (
    meetings_df.dropna(subset=["MeetingTopic_norm", "Year"])
               .drop_duplicates(subset=["MeetingTopic_norm"])
               .set_index("MeetingTopic_norm")["Year"]
               .to_dict()
)

# Attach year onto participants using normalized title
participants_df["Year"] = participants_df["Meeting_norm"].map(meeting_to_year)

participants_df[["Participant", "Meeting", "Year"]]

,Participant,Meeting,Year
0,"Alison Ringel, PhD",Targeting Lipid Biology in Cancer,2023.0
1,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer,2023.0
2,"Bart Vanhaesebroeck, PhD",Targeting Lipid Biology in Cancer,2023.0
3,"Christina Mitchell, MB BS, PhD",Targeting Lipid Biology in Cancer,2023.0
4,"Neil Vasan, MD, PhD",Targeting Lipid Biology in Cancer,2023.0
...,...,...,...
2375,"Jeffrey Ward, MD, PhD",Uncovering new mechanisms of LKB1-mediated tum...,2023.0
2376,"David Barbie, MD",Uncovering new mechanisms of LKB1-mediated tum...,2023.0
2377,"David Shackelford, PhD",Uncovering new mechanisms of LKB1-mediated tum...,2023.0
2378,"Daniel Frigo, PhD",Uncovering new mechanisms of LKB1-mediated tum...,2023.0


In [7]:
# Cell 4 — (1) create dict keyed by "Meeting Topic (Year)" with list of participant full names

# Keep only rows that have a meeting + participant name
p = participants_df.dropna(subset=["Meeting_norm", "Participant"]).copy()

# Build a key string like "Targeting Lipid Biology in Cancer (2021)"
def make_meeting_year_key(meeting_norm, year):
    y = "" if pd.isna(year) else str(int(year)) if float(year).is_integer() else str(year)
    return f"{meeting_norm} ({y})" if y else f"{meeting_norm} (Year Unknown)"

p["MeetingYearKey"] = [make_meeting_year_key(m, y) for m, y in zip(p["Meeting_norm"], p["Year"])]

meeting_attendees_dict = (
    p.groupby("MeetingYearKey")["Participant"]
     .apply(lambda s: sorted(set(s.dropna().astype(str).str.strip())))
     .to_dict()
)

# Example: show first 5 keys
list(meeting_attendees_dict["Targeting Lipid Biology in Cancer (2023)"])

['Alison Ringel, PhD',
 'Bart Vanhaesebroeck, PhD',
 'Brooke Emerling, PhD',
 'Christina Mitchell, MB BS, PhD',
 'David Fruman, PhD',
 'Emilio Hirsch, PhD',
 'Gretchen Alicea, PhD',
 'Hua Eleanor Yu, PhD',
 'Jeremy Baskin, PhD',
 'Karen Dixon,',
 'Livia  Schiavinato Eberlin, PhD',
 'Neil Vasan, MD, PhD',
 'Prof Banafshe  Larijani , PhD',
 'Ray Blind,',
 'Sarah  Skuli,',
 'Tamas Balla, MD, PhD',
 'Targeting Lipid Biology in Cancer',
 'Vytas Bankaitis, PhD']

In [8]:
# --- NIH RePORTER: query by PI names pulled from meeting_attendees_dict ---


REPORTER_SEARCH_URL = "https://api.reporter.nih.gov/v2/projects/search"


# ---------- name parsing ----------

_SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "md", "phd", "mph", "ms", "m.d.", "ph.d.", "dr"}
def _clean_person_name(name: str) -> str:
    if name is None:
        return ""
    s = re.sub(r"\s+", " ", str(name)).strip()
    # drop parenthetical stuff
    s = re.sub(r"\([^)]*\)", "", s).strip()
    # drop credentials after commas, but keep "Last, First" structure if present
    # If there are 2+ commas, it's probably "Last, First, MD, PhD" -> keep first two parts
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if len(parts) >= 2:
        s = f"{parts[0]}, {parts[1]}"
    elif len(parts) == 1:
        s = parts[0]
    return s

def parse_first_last(name: str) -> tuple[str, str]:
    """
    Returns (first_name, last_name) when possible.
    Handles:
      - "Last, First"
      - "First Last"
      - "First M. Last"
      - "Dr. First Last, MD" (credentials stripped)
    """
    s = _clean_person_name(name)
    if not s:
        return "", ""

    # remove common honorifics
    s = re.sub(r"^(dr\.?|prof\.?|mr\.?|ms\.?|mrs\.?)\s+", "", s, flags=re.I).strip()

    if "," in s:  # "Last, First"
        last, first = [p.strip() for p in s.split(",", 1)]
        first_tokens = [t for t in first.split() if t]
        # remove suffix tokens
        first_tokens = [t for t in first_tokens if t.lower().strip(".") not in _SUFFIXES]
        first_name = first_tokens[0] if first_tokens else ""
        return first_name, last

    # "First ... Last"
    tokens = [t for t in s.split() if t]
    tokens = [t for t in tokens if t.lower().strip(".") not in _SUFFIXES]
    if len(tokens) == 1:
        # single token; treat as last name
        return "", tokens[0]
    return tokens[0], tokens[-1]


def build_pi_names_criteria(names: list[str]) -> list[dict]:
    """
    Builds the RePORTER criteria.pi_names list.
    Example output:
      [{"any_name":"gullo","first_name":"michael"}, {"any_name":"welch","first_name":"brian"}]
    We include last name in any_name, and (if available) first_name.
    """
    pis = []
    seen = set()

    for n in names:
        first, last = parse_first_last(n)
        if not last:
            continue

        key = (first.lower(), last.lower())
        if key in seen:
            continue
        seen.add(key)

        entry = {"any_name": last.lower(), "first_name": first.lower() if first else ""}
        pis.append(entry)

    return pis


# ---------- RePORTER paging + award sums ----------

def reporter_search_all_pages(
    base_params: dict,
    freeze_path: Path | None = None,
    sleep_s: float = 0.3,
    limit: int = 500,
) -> dict:
    """
    Executes a RePORTER v2 projects/search request and paginates until all results fetched.
    base_params is the *full* payload you want to send (will be deep-copied).
    Returns {"total": int, "pages": [page_json, ...]}.
    If freeze_path is provided, saves each page as JSON on its own line (JSONL).
    """
    params = deepcopy(base_params)
    params.setdefault("limit", limit)
    params["offset"] = 0

    # If frozen exists, replay from disk
    if freeze_path is not None and freeze_path.exists():
        loaded = []
        with freeze_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("Date accessed:"):
                    continue
                loaded.append(json.loads(line))
        total = loaded[0].get("meta", {}).get("total", 0) if loaded else 0
        return {"total": total, "pages": loaded}

    session = requests.Session()
    pages = []
    total = None

    while True:
        r = session.post(REPORTER_SEARCH_URL, json=params, timeout=60)
        r.raise_for_status()
        page = r.json()
        pages.append(page)

        if total is None:
            total = page.get("meta", {}).get("total", 0)

        if freeze_path is not None:
            freeze_path.parent.mkdir(parents=True, exist_ok=True)
            with freeze_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(page))
                f.write("\n")

        offset = page.get("meta", {}).get("offset", params["offset"])
        returned = page.get("meta", {}).get("count", len(page.get("results", [])))
        next_offset = offset + returned

        if returned == 0 or next_offset >= total:
            break

        params["offset"] = next_offset
        time.sleep(sleep_s)

    if freeze_path is not None:
        with freeze_path.open("a", encoding="utf-8") as f:
            f.write(f"Date accessed: {datetime.now().isoformat()}\n")

    return {"total": int(total or 0), "pages": pages}


def sum_award_amounts_from_pages(pages: list[dict]) -> int:
    """
    Sums award amounts across returned projects.
    Adjust field choice if your response differs.
    """
    total_cost = 0
    for page in pages:
        for proj in page.get("results", []):
            cost = (
                proj.get("fy_total_cost")
                if proj.get("fy_total_cost") is not None
                else proj.get("award_amount")
            )
            if cost is None:
                cost = 0
            total_cost += int(cost)
    return total_cost


# ---------- main driver: meeting_attendees_dict -> PI query ----------

def reporter_grants_by_meeting_pis(
    meeting_attendees_dict: dict,
    NIH_param_template: dict,
    raw_dir: str | Path | None = None,
    sleep_s: float = 0.3,
) -> pd.DataFrame:
    """
    For each meeting in meeting_attendees_dict:
      - builds criteria.pi_names from attendee names
      - runs NIH RePORTER search
      - returns per-meeting summary + per-PI breakdown (optional)
    Output columns include MeetingYearKey, total grants, total award, and the PI list used.

    Notes:
    - RePORTER pi_names criteria searches PIs; it does not guarantee "applied to their names"
      equals "awarded to them", but it’s the standard PI-based filter in the API.
    """
    raw_dir = Path(raw_dir) if raw_dir is not None else None
    rows = []

    for meeting_key, names in meeting_attendees_dict.items():
        pi_names = build_pi_names_criteria(names)

        # If no parsable names, skip
        if not pi_names:
            rows.append({
                "MeetingYearKey": meeting_key,
                "PI_count_used": 0,
                "Grant_total": 0,
                "Award_total": 0,
                "PI_names_payload": [],
            })
            continue

        payload = deepcopy(NIH_param_template)
        payload.setdefault("criteria", {})
        payload["criteria"]["pi_names"] = pi_names

        # Optional: if your template includes advanced_text_search, leave it as-is;
        # you can also remove it to ensure you're ONLY filtering by PI names:
        payload["criteria"].pop("advanced_text_search", None)

        freeze_path = None
        if raw_dir is not None:
            safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", meeting_key)[:180]
            freeze_path = raw_dir / f"{safe_name}_NIH_pi_search.jsonl"

        res = reporter_search_all_pages(
            base_params=payload,
            freeze_path=freeze_path,
            sleep_s=sleep_s
        )
        grant_total = int(res["total"] or 0)
        award_total = sum_award_amounts_from_pages(res["pages"]) if grant_total else 0

        rows.append({
            "MeetingYearKey": meeting_key,
            "PI_count_used": len(pi_names),
            "Grant_total": grant_total,
            "Award_total": award_total,
            "PI_names_payload": pi_names,   # keep for traceability
        })

        # be polite to API
        time.sleep(sleep_s)

    return pd.DataFrame(rows)

In [9]:

meeting_pi_df = reporter_grants_by_meeting_pis(
    meeting_attendees_dict=meeting_attendees_dict,
    NIH_param_template=NIH_param,
    raw_dir="raw_reporter_freezes_by_meeting",  # set None to disable freezing
    sleep_s=0.3
)

meeting_pi_df.sort_values("Grant_total", ascending=False).head(10)

NameError: name 'NIH_param' is not defined